# W&B Experiment Tracking — Live Training of a TinyTransolver

This notebook trains a **~811k-parameter TinyTransolver** — a
faithful, downscaled version of the base architecture PhysicsX's
PXtransolver builds on — on the synthetic NACA airfoil dataset that
`registry.ipynb` promoted into the W&B Registry. Along the way you
will see:

1. A training loop that logs loss metrics to W&B in real time.
2. Per-epoch model checkpoints saved as versioned artifacts.
3. The best-epoch checkpoint linked into the `airfoil_surrogates`
   Registry under the `checkpoints` Collection.
4. W&B Tables, custom charts, and Transolver-specific custom metrics —
   logged inline as training happens.

The training run is opened inside a `with wandb.init(...) as run:`
block so it finishes cleanly on exit.

**Prerequisites**
- `uv sync` (installs `wandb`, `torch`, `numpy`, `matplotlib` per `pyproject.toml`).
- `uv run wandb login`.
- **`registry.ipynb` executed first** — it produces
  `wandb-registry-airfoil_surrogates/datasets:production`, which this
  notebook consumes at its setup step.
- Access to the `wandb_emea` entity on W&B, or edit the `ENTITY`
  constant in §4 below.

**Background reading**
- Physics-Attention and the base Transolver: `docs/explainers/pxtransolver.md`.
- W&B Registry deep dive: `notebooks/archive/registry_tutorial.ipynb`.


## 1. Setup

Third-party imports and the `AirfoilCase` dataclass used to typed-load
the Registry-served dataset. The synthetic dataset generator itself
lives in `registry.ipynb`.


In [1]:
from __future__ import annotations

import json
import math
import tempfile
from dataclasses import dataclass
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import wandb

matplotlib.use("Agg")                # headless plotting; wandb.Image takes figs
device = torch.device("cpu")          # notebook is CPU-friendly

print(f"torch  = {torch.__version__}")
print(f"wandb  = {wandb.__version__}")
print(f"device = {device}")


torch  = 2.12.1
wandb  = 0.28.0
device = cpu


In [ ]:
# ---- Data utility: AirfoilCase dataclass ----
# The synthetic generator lives in `registry.ipynb`. This notebook
# only needs the AirfoilCase container to load the Registry-served
# JSON back into typed objects with .features() / .targets() methods.


@dataclass
class AirfoilCase:
    points: np.ndarray    # (N, 2) - xy positions in the flow domain
    sdf: np.ndarray       # (N,)   - signed distance to the airfoil surface
    aoa: float            # angle of attack (radians)
    re: float             # Reynolds number
    pressure: np.ndarray  # (N,)   - target pressure coefficient
    velocity: np.ndarray  # (N, 2) - target (u, v)

    def features(self) -> np.ndarray:
        """Per-point input features: (x, y, sdf, aoa, log_re)."""
        n = self.points.shape[0]
        aoa_col = np.full((n, 1), self.aoa, dtype=np.float32)
        re_col = np.full((n, 1), np.log10(self.re) - 6.0, dtype=np.float32)
        return np.concatenate(
            [self.points.astype(np.float32),
             self.sdf[:, None].astype(np.float32),
             aoa_col, re_col],
            axis=1,
        )

    def targets(self) -> np.ndarray:
        """Per-point targets: (pressure, u, v)."""
        return np.concatenate(
            [self.pressure[:, None], self.velocity], axis=1
        ).astype(np.float32)


## 2. Data — from the W&B Registry

The synthetic NACA airfoil dataset was generated and promoted to the
Registry by `registry.ipynb`. This notebook consumes it via
`wandb.Api().artifact("wandb-registry-airfoil_surrogates/datasets:production")`
— the same path any downstream consumer (sweeps, reports, ...) uses.

Each `AirfoilCase` is a point-cloud snapshot of a NACA-4-digit airfoil
under one `(angle_of_attack, Reynolds_number)`. Features per point:
`(x, y, sdf, aoa, log_re)`. Targets per point: `(pressure, u, v)`.
Payload is JSON so it's easy to inspect.


In [ ]:
# ---- Consume the dataset from the W&B Registry ----
REGISTRY_NAME = "airfoil_surrogates"
DATASET_COLLECTION = "datasets"
DATASET_REGISTRY_PATH = f"wandb-registry-{REGISTRY_NAME}/{DATASET_COLLECTION}:production"
VAL_FRACTION = 0.2

api = wandb.Api()
art = api.artifact(DATASET_REGISTRY_PATH)
ds_dir = Path(art.download())
json_file = next(ds_dir.glob("*.json"))
with open(json_file) as f:
    payload = json.load(f)

cases = [
    AirfoilCase(
        points=np.array(c["points"], dtype=np.float32),
        sdf=np.array(c["sdf"], dtype=np.float32),
        aoa=float(c["aoa"]),
        re=float(c["re"]),
        pressure=np.array(c["pressure"], dtype=np.float32),
        velocity=np.array(c["velocity"], dtype=np.float32),
    )
    for c in payload["cases"]
]

n_val = int(len(cases) * VAL_FRACTION)
train_cases = cases[:-n_val]
val_cases = cases[-n_val:]

torch.manual_seed(42)                                     # deterministic training

print(f"pulled {DATASET_REGISTRY_PATH}")
print(f"dataset meta: {payload['meta']}")
print(f"total cases: {len(cases)}   train: {len(train_cases)}   val: {len(val_cases)}")

sample = train_cases[0]
print()
print("One AirfoilCase (what the model sees):")
print(f"  features shape : {sample.features().shape}   (per point: [x, y, sdf, aoa, log_re])")
print(f"  targets  shape : {sample.targets().shape}    (per point: [pressure, u, v])")
print(f"  angle of attack: {math.degrees(sample.aoa):.2f} deg")
print(f"  Reynolds number: {sample.re:.1f}")


## 3. The model — a tiny (~811k-parameter) Transolver

We define the model **inline** so you can read the Physics-Attention
mechanism straight through the notebook. There are three concepts to
hold in your head; the code below implements them in the order they
run.

**Slicing.** Each mesh point `x_i` (with `C` hidden channels) is softly
assigned to `M` "physics slices" via a learned linear projection
followed by softmax:

```
w_i = softmax(Project(x_i))       # w_i has shape (M,), sums to 1
```

Each slice `j` aggregates the point features into a single token by a
weighted mean:

```
z_j = sum_i w_{i,j} * x_i  /  sum_i w_{i,j}
```

**Attention.** Standard multi-head scaled-dot-product attention runs
on the `M` slice tokens (not on all `N` mesh points):

```
z' = softmax(Q K^T / sqrt(d)) * V
```

**De-slicing.** Each updated slice token is broadcast back to the mesh
points using the same membership weights:

```
x'_i = sum_j w_{i,j} * z'_j
```

Complexity is `O(N * M * C + M^2 * C)` — linear in `N` because `M <<
N`. This is the trick that makes Transolver scale to million-point
meshes.

Full mathematical treatment: `docs/explainers/pxtransolver.md`.


In [4]:
class PhysicsAttention(nn.Module):
    """One Physics-Attention block: slice -> attend -> deslice, plus an FFN."""

    def __init__(self, hidden: int, n_slices: int, n_heads: int, ffn_expansion: int = 2):
        super().__init__()
        assert hidden % n_heads == 0, "hidden must be divisible by n_heads"
        self.n_slices = n_slices
        self.n_heads = n_heads
        self.head_dim = hidden // n_heads

        # Slicing projection: for each point, produce M slice-assignment logits.
        self.slice_proj = nn.Linear(hidden, n_slices)

        # QKV for multi-head attention on the M slice tokens.
        self.qkv = nn.Linear(hidden, 3 * hidden)
        self.attn_out = nn.Linear(hidden, hidden)

        # Position-wise feed-forward.
        self.ffn = nn.Sequential(
            nn.Linear(hidden, ffn_expansion * hidden),
            nn.GELU(),
            nn.Linear(ffn_expansion * hidden, hidden),
        )

        self.norm1 = nn.LayerNorm(hidden)
        self.norm2 = nn.LayerNorm(hidden)

    def _slice_attend_deslice(self, x: torch.Tensor):
        # x: (B, N, C)
        B, N, C = x.shape
        M, H, D = self.n_slices, self.n_heads, self.head_dim

        # 1. Slicing: soft assignments of each point to each of M slices.
        w = torch.softmax(self.slice_proj(x), dim=-1)             # (B, N, M)

        # 2. Aggregate points into M slice tokens.
        numer = torch.einsum("bnm,bnc->bmc", w, x)                 # (B, M, C)
        denom = w.sum(dim=1).unsqueeze(-1) + 1e-8                  # (B, M, 1)
        z = numer / denom                                           # (B, M, C)

        # 3. Multi-head attention over slice tokens.
        qkv = self.qkv(z).view(B, M, 3, H, D).permute(2, 0, 3, 1, 4)  # (3, B, H, M, D)
        q, k, v = qkv[0], qkv[1], qkv[2]
        scores = (q @ k.transpose(-2, -1)) / math.sqrt(D)               # (B, H, M, M)
        attn = torch.softmax(scores, dim=-1)
        z_out = (attn @ v).transpose(1, 2).contiguous().view(B, M, C)   # (B, M, C)
        z_out = self.attn_out(z_out)

        # 4. Deslicing: broadcast slice tokens back to N points via same weights.
        x_out = torch.einsum("bnm,bmc->bnc", w, z_out)                  # (B, N, C)
        return x_out, w

    def forward(self, x: torch.Tensor, return_slice_weights: bool = False):
        attended, w = self._slice_attend_deslice(self.norm1(x))
        x = x + attended
        x = x + self.ffn(self.norm2(x))
        if return_slice_weights:
            return x, w
        return x


In [5]:
class TinyTransolver(nn.Module):
    """Small Transolver: linear embed + L Physics-Attention blocks + output head."""

    def __init__(
        self,
        in_dim: int = 5,
        out_dim: int = 3,
        hidden: int = 128,
        n_layers: int = 6,
        n_slices: int = 32,
        n_heads: int = 8,
        ffn_expansion: int = 2,
    ):
        super().__init__()
        self.embed = nn.Linear(in_dim, hidden)
        self.blocks = nn.ModuleList([
            PhysicsAttention(hidden, n_slices, n_heads, ffn_expansion)
            for _ in range(n_layers)
        ])
        self.out_norm = nn.LayerNorm(hidden)
        self.head = nn.Linear(hidden, out_dim)

    def forward(self, features: torch.Tensor, return_slice_weights: bool = False):
        x = self.embed(features)                     # (B, N, hidden)
        all_w = []
        for block in self.blocks:
            if return_slice_weights:
                x, w = block(x, return_slice_weights=True)
                all_w.append(w)
            else:
                x = block(x)
        x = self.out_norm(x)
        out = self.head(x)                            # (B, N, out_dim)
        if return_slice_weights:
            return out, all_w                         # list of L (B, N, M) tensors
        return out


def count_params(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


In [6]:
# ---- Instantiate and sanity-check ----
MODEL_CONFIG = {
    "in_dim": 5,
    "out_dim": 3,
    "hidden": 128,
    "n_layers": 6,
    "n_slices": 32,
    "n_heads": 8,
    "ffn_expansion": 2,
}

model = TinyTransolver(**MODEL_CONFIG).to(device)
print(model)
print()
print(f"parameter count: {count_params(model):,}")

with torch.no_grad():
    dummy_features = torch.from_numpy(train_cases[0].features()).float().unsqueeze(0)  # (1, N, 5)
    dummy_out = model(dummy_features)
    print(f"forward pass:   in={tuple(dummy_features.shape)} -> out={tuple(dummy_out.shape)}")


TinyTransolver(
  (embed): Linear(in_features=5, out_features=128, bias=True)
  (blocks): ModuleList(
    (0-5): 6 x PhysicsAttention(
      (slice_proj): Linear(in_features=128, out_features=32, bias=True)
      (qkv): Linear(in_features=128, out_features=384, bias=True)
      (attn_out): Linear(in_features=128, out_features=128, bias=True)
      (ffn): Sequential(
        (0): Linear(in_features=128, out_features=256, bias=True)
        (1): GELU(approximate='none')
        (2): Linear(in_features=256, out_features=128, bias=True)
      )
      (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True, bias=True)
      (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True, bias=True)
    )
  )
  (out_norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True, bias=True)
  (head): Linear(in_features=128, out_features=3, bias=True)
)

parameter count: 821,059
forward pass:   in=(1, 512, 5) -> out=(1, 512, 3)


## 4. Train the model and promote the best checkpoint

One W&B run covers the whole training session: consume the dataset
from the Registry (creating a lineage edge), train the Transolver,
log per-epoch metrics + checkpoints + visualisations, ensure the
custom Registry exists, and link the best-val checkpoint into it.
Everything inside one `with wandb.init(job_type="train") as run:`
block.

**Data preparation before the loop.** We compute per-channel `mean` and
`std` on the **training split only** and standard-normalise both
features and targets. Training MSE is measured in std-units, so
pressure and velocity contribute comparably to the loss. Predictions
are denormalised only when we render the pressure-field plots for the
`wandb.Table`.

**Loss.** Two terms.

- **`mse`** — mean-squared error on the normalised targets. The
  workhorse regression loss.
- **`aux`** — load-balancing penalty on the Physics-Attention slicing.
  For each of the six routing blocks it combines *two* sub-terms:
  - **Batch balance**: `-entropy(batch-averaged slice usage)`. Pushes
    the mean usage across the batch toward uniform. Guards against
    mode collapse (all points into one slice).
  - **Per-point sharpness**: `entropy(per-point softmax)`. Pushes
    each point to commit to a small number of slices. Guards against
    the degenerate "everything uniform" solution the balance term
    alone would allow (which is what happens if you only reward
    batch balance — the model shrinks `slice_proj` toward zero and
    routes every point uniformly).

  `total_loss = mse + λ_aux * aux` with `λ_aux = 0.01`. The minimum
  of `aux` is at the healthy configuration — sharp per-point
  assignments *and* uniform batch usage.

**Per-epoch work.**

1. **Train step.** Log `train/loss`, `train/mse`, `train/aux` per batch.
2. **Val step.** Log a full metric set:
   - `val/loss` — MSE on normalised val targets.
   - `val/pressure_rmse`, `val/velocity_rmse`, `val/u_rmse`,
     `val/v_rmse` — per-channel RMSEs in std-units, so all four are
     directly comparable.
   - `val/slice_entropy` — per-point average entropy of last-block
     slice weights. Low = model committed to sharp per-point
     assignments.
   - `val/slice_usage_entropy` — entropy of the *batch-averaged*
     slice usage. High = all 32 slices see some points (healthy).
     Low = mode collapse. This is the metric that actually catches
     the failure mode `val/slice_entropy` misses.
3. **Checkpoint.** Save model state; log as a new version of
   `airfoil_surrogate_ckpt`, aliased `epoch-N`.
4. **Every 3 epochs** log richer visuals — a `wandb.Table` with
   inline pressure-field images (predictions denormalised for
   physical interpretability), a `wandb.plot.bar` chart of
   normalised pressure RMSE by AoA bucket, and a slice-segmentation
   `wandb.Image` for one canonical val case.

After the loop the same run:

- Ensures the `airfoil_surrogates` Registry exists (idempotent via
  try/except — usually created already by `registry.ipynb`).
- Waits for the best-val checkpoint to finish uploading via the
  reference stashed during the loop (`best_ckpt_artifact.wait()`) —
  `log_artifact` is async, so we don't try to look up the alias
  before it's committed on the backend.
- Calls `run.link_artifact(best_ckpt_artifact, target_path=...,
  aliases=["production", "epoch-N"])` — the `checkpoints` Collection
  is created implicitly on first link.

Watch the W&B UI in a separate tab.


In [9]:
# ---- Helpers + section-local training config ----
def stack_cases(cases_list):
    X = torch.stack([torch.from_numpy(c.features()).float() for c in cases_list])
    Y = torch.stack([torch.from_numpy(c.targets()).float() for c in cases_list])
    return X, Y


def pressure_figure(case, prediction):
    """Side-by-side scatter of truth vs predicted pressure for one AirfoilCase."""
    fig, axes = plt.subplots(1, 2, figsize=(6, 3), dpi=90)
    for ax, values, title in zip(axes, [case.pressure, prediction[:, 0]], ["truth", "pred"]):
        sc = ax.scatter(case.points[:, 0], case.points[:, 1], c=values, cmap="RdBu_r", s=6)
        ax.set_title(f"Cp - {title}")
        ax.set_aspect("equal")
        ax.axis("off")
        plt.colorbar(sc, ax=ax, fraction=0.04)
    fig.tight_layout()
    return fig


def slice_figure(case, slice_argmax, epoch):
    fig, ax = plt.subplots(figsize=(3.5, 3), dpi=90)
    ax.scatter(case.points[:, 0], case.points[:, 1], c=slice_argmax, cmap="tab20", s=10)
    ax.set_title(f"slice assignment - epoch {epoch}")
    ax.set_aspect("equal")
    ax.axis("off")
    fig.tight_layout()
    return fig


TRAIN_CONFIG = {
    **MODEL_CONFIG,
    "epochs": 12,
    "batch_size": 8,
    "lr": 3e-3,
    "weight_decay": 1e-4,
    "lambda_aux": 0.01,                # weight on the load-balancing aux loss
    "n_cases": len(train_cases),
    "n_points": N_POINTS,
    "seed": SEED,
    "n_params": count_params(model),
}

X_train, Y_train = stack_cases(train_cases)
X_val, Y_val = stack_cases(val_cases)
print(f"X_train shape: {tuple(X_train.shape)}    Y_train shape: {tuple(Y_train.shape)}")
print(f"X_val   shape: {tuple(X_val.shape)}    Y_val   shape: {tuple(Y_val.shape)}")


X_train shape: (64, 512, 5)    Y_train shape: (64, 512, 3)
X_val   shape: (16, 512, 5)    Y_val   shape: (16, 512, 3)


In [ ]:
# ---- Section-local variables ----
ENTITY = "wandb_emea"
PROJECT = "jb_physics_demo"

DATASET_REGISTRY_PATH = f"wandb-registry-{REGISTRY_NAME}/{DATASET_COLLECTION}:production"
CHECKPOINT_ARTIFACT_NAME = "airfoil_surrogate_ckpt"

CHECKPOINT_COLLECTION = "checkpoints"


with wandb.init(
    entity=ENTITY,
    project=PROJECT,
    job_type="train",
    config=TRAIN_CONFIG,
    tags=["notebook", "experiment-tracking", "tiny-transolver"],
    name="tiny-transolver-train",
) as run:
    # Consume the Registry-served dataset - creates the dataset -> training lineage edge.
    run.use_artifact(DATASET_REGISTRY_PATH)

    # --- Normalisation stats (train-set only, applied to both splits) ---
    # Both features and targets get standard-normalised. Targets end up with unit
    # std per channel, so pressure and velocity RMSEs are directly comparable.
    feat_mean = X_train.mean(dim=(0, 1), keepdim=True)                  # (1, 1, 5)
    feat_std = X_train.std(dim=(0, 1), keepdim=True).clamp(min=1e-6)
    tgt_mean = Y_train.mean(dim=(0, 1), keepdim=True)                   # (1, 1, 3)
    tgt_std = Y_train.std(dim=(0, 1), keepdim=True).clamp(min=1e-6)

    X_train_n = (X_train - feat_mean) / feat_std
    Y_train_n = (Y_train - tgt_mean) / tgt_std
    X_val_n = (X_val - feat_mean) / feat_std
    Y_val_n = (Y_val - tgt_mean) / tgt_std

    tgt_mean_dev = tgt_mean.to(device)
    tgt_std_dev = tgt_std.to(device)

    print("target normalisation stats (channels: [pressure, u, v]):")
    print(f"  mean = {tgt_mean.squeeze().tolist()}")
    print(f"  std  = {tgt_std.squeeze().tolist()}")

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=TRAIN_CONFIG["lr"],
        weight_decay=TRAIN_CONFIG["weight_decay"],
    )

    LAMBDA_AUX = TRAIN_CONFIG["lambda_aux"]
    best_val_rmse = float("inf")
    best_epoch = None
    best_ckpt_artifact = None
    n = X_train_n.size(0)
    batch_size = TRAIN_CONFIG["batch_size"]

    for epoch in range(TRAIN_CONFIG["epochs"]):
        # --- train ---
        model.train()
        perm = torch.randperm(n)
        train_losses = []
        for start in range(0, n, batch_size):
            idx = perm[start:start + batch_size]
            xb = X_train_n[idx].to(device)
            yb = Y_train_n[idx].to(device)
            pred, all_w = model(xb, return_slice_weights=True)
            mse = F.mse_loss(pred, yb)

            # Two-term aux loss (applied at every routing block).
            # Term A - batch balance:  minimise `-entropy(batch-averaged usage)`
            #   -> pushes toward uniform slice usage across the batch, preventing
            #      mode collapse where all points route to the same slice.
            # Term B - per-point sharpness:  minimise `entropy(per-point softmax)`
            #   -> pushes toward each point committing to a small number of
            #      slices, preventing the degenerate "everything uniform"
            #      solution the balance term alone would allow.
            aux_terms = []
            for w in all_w:
                p_slice = w.mean(dim=(0, 1)).clamp(min=1e-8)                # (M,)
                usage_entropy_j = -(p_slice * p_slice.log()).sum()
                balance_term = -usage_entropy_j                              # low when uniform

                w_clamp = w.clamp(min=1e-8)
                per_pt_entropy_j = -(w_clamp * w_clamp.log()).sum(dim=-1).mean()
                sharpness_term = per_pt_entropy_j                            # low when sharp

                aux_terms.append(balance_term + sharpness_term)
            aux = torch.stack(aux_terms).mean()

            loss = mse + LAMBDA_AUX * aux
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())
            wandb.log({
                "train/loss": loss.item(),
                "train/mse": mse.item(),
                "train/aux": aux.item(),
                "epoch": epoch,
            })

        train_loss_epoch = float(np.mean(train_losses))

        # --- validate ---
        model.eval()
        with torch.no_grad():
            xv = X_val_n.to(device)
            yv = Y_val_n.to(device)
            pred_n, all_w = model(xv, return_slice_weights=True)
            val_mse = F.mse_loss(pred_n, yv).item()

            # Aux term computed identically to training so train/val loss
            # curves are directly comparable.
            val_aux_terms = []
            for w in all_w:
                p_slice = w.mean(dim=(0, 1)).clamp(min=1e-8)
                balance_term = -(-(p_slice * p_slice.log()).sum())              # -usage_entropy
                w_clamp = w.clamp(min=1e-8)
                sharpness_term = -(w_clamp * w_clamp.log()).sum(dim=-1).mean()  # per-pt entropy
                val_aux_terms.append(balance_term + sharpness_term)
            val_aux = torch.stack(val_aux_terms).mean().item()
            val_loss = val_mse + LAMBDA_AUX * val_aux

            # Per-channel normalised RMSEs. Targets have unit std per channel,
            # so pressure and velocity RMSEs are directly comparable.
            err = pred_n - yv                                              # (B, N, 3)
            pressure_rmse = err[..., 0].pow(2).mean().sqrt().item()
            u_rmse = err[..., 1].pow(2).mean().sqrt().item()
            v_rmse = err[..., 2].pow(2).mean().sqrt().item()
            velocity_rmse = float(((u_rmse ** 2 + v_rmse ** 2) / 2) ** 0.5)

            # Slice-health diagnostics.
            last_w = all_w[-1]                                             # (B, N, M)
            p_pt = last_w.clamp(min=1e-8)
            slice_entropy = -(p_pt * p_pt.log()).sum(dim=-1).mean().item() # per-point avg
            p_batch = last_w.mean(dim=(0, 1)).clamp(min=1e-8)              # (M,)
            slice_usage_entropy = -(p_batch * p_batch.log()).sum().item()  # batch-level

            # Denormalise predictions for physical-space visualisation.
            val_pred = pred_n * tgt_std_dev + tgt_mean_dev

        wandb.log({
            "epoch": epoch,
            "train/epoch_loss": train_loss_epoch,
            "val/loss": val_loss,      # mse + lambda*aux, same objective as train/loss
            "val/mse": val_mse,        # task metric alone, comparable with train/mse
            "val/aux": val_aux,        # regulariser diagnostic on val
            "val/pressure_rmse": pressure_rmse,
            "val/velocity_rmse": velocity_rmse,
            "val/u_rmse": u_rmse,
            "val/v_rmse": v_rmse,
            "val/slice_entropy": slice_entropy,
            "val/slice_usage_entropy": slice_usage_entropy,
        })

        # --- checkpoint as a new artifact version ---
        ckpt_path = tmp_dir / "tiny_transolver.pt"
        torch.save(
            {
                "state_dict": model.state_dict(),
                "config": dict(TRAIN_CONFIG),
                "epoch": epoch,
                "val_pressure_rmse": pressure_rmse,
            },
            ckpt_path,
        )
        ckpt_artifact = wandb.Artifact(
            name=CHECKPOINT_ARTIFACT_NAME,
            type="model",
            metadata={
                "epoch": epoch,
                "val_pressure_rmse": pressure_rmse,
                "val_slice_entropy": slice_entropy,
                "n_params": count_params(model),
            },
        )
        ckpt_artifact.add_file(str(ckpt_path))
        # log_artifact() is async: it returns a reference we can .wait() on later.
        logged = run.log_artifact(ckpt_artifact, aliases=[f"epoch-{epoch}"])

        if pressure_rmse < best_val_rmse:
            best_val_rmse = pressure_rmse
            best_epoch = epoch
            best_ckpt_artifact = logged

        # --- richer visuals every 3 epochs and on the final epoch ---
        if epoch % 3 == 0 or epoch == TRAIN_CONFIG["epochs"] - 1:
            pred_n_np = pred_n.cpu().numpy()                     # normalised, for per-case RMSE
            val_pred_np = val_pred.cpu().numpy()                 # denormalised, for plotting
            y_val_n_np = Y_val_n.numpy()

            # Table: per-case pressure_rmse in NORMALISED space (comparable across cases),
            # side by side with a physical-space pressure-field plot.
            table = wandb.Table(columns=["case_id", "aoa_deg", "re", "pressure_rmse_norm", "field"])
            for i in range(min(6, len(val_cases))):
                c = val_cases[i]
                rmse_i = float(np.sqrt(np.mean((pred_n_np[i, :, 0] - y_val_n_np[i, :, 0]) ** 2)))
                fig = pressure_figure(c, val_pred_np[i])
                table.add_data(i, float(math.degrees(c.aoa)), c.re, rmse_i, wandb.Image(fig))
                plt.close(fig)
            wandb.log({f"val/predictions_epoch_{epoch}": table})

            # Custom bar chart: normalised pressure RMSE by AoA bucket.
            buckets = [("0-5 deg", 0.0, 5.0), ("5-10 deg", 5.0, 10.0), ("10-15 deg", 10.0, 15.0)]
            rows = []
            for label, lo, hi in buckets:
                in_bucket = [
                    float(np.sqrt(np.mean((pred_n_np[i, :, 0] - y_val_n_np[i, :, 0]) ** 2)))
                    for i, c in enumerate(val_cases)
                    if lo <= math.degrees(c.aoa) < hi
                ]
                if in_bucket:
                    rows.append([label, float(np.mean(in_bucket))])
            if rows:
                bucket_table = wandb.Table(data=rows, columns=["aoa_bucket", "mean_rmse_norm"])
                wandb.log({
                    f"val/rmse_by_aoa_epoch_{epoch}": wandb.plot.bar(
                        bucket_table,
                        "aoa_bucket",
                        "mean_rmse_norm",
                        title=f"Val pressure RMSE (normalised) by AoA bucket - epoch {epoch}",
                    )
                })

            # Slice-segmentation image on one canonical val case (last block).
            canonical = val_cases[0]
            slice_argmax = last_w[0].argmax(dim=-1).cpu().numpy()
            fig_slice = slice_figure(canonical, slice_argmax, epoch)
            wandb.log({f"model/slice_assignment_epoch_{epoch}": wandb.Image(fig_slice)})
            plt.close(fig_slice)

        print(
            f"epoch {epoch:2d} | train {train_loss_epoch:.4f} | "
            f"p_rmse {pressure_rmse:.4f} | v_rmse {velocity_rmse:.4f} | "
            f"slice_H {slice_entropy:.3f} | usage_H {slice_usage_entropy:.3f}"
        )

    print()
    print(f"best epoch: {best_epoch}   val_pressure_rmse: {best_val_rmse:.4f}")

    # --- Ensure the custom Registry exists + link best checkpoint (still inside the run) ---
    api = wandb.Api()

    try:
        registry = api.create_registry(
            name=REGISTRY_NAME,
            visibility="organization",
        )
        print(f"created registry: {registry.name}")
    except Exception as exc:
        print(f"registry create skipped (may already exist): {exc}")

    # log_artifact() is async - wait for the best checkpoint to finish uploading
    # before we link it into the Registry. Using the reference we stashed
    # during the loop avoids an api.artifact() lookup that would 404 on the
    # not-yet-committed alias.
    best_alias = f"epoch-{best_epoch}"
    best_ckpt_artifact.wait()

    target_path = f"wandb-registry-{REGISTRY_NAME}/{CHECKPOINT_COLLECTION}"
    run.link_artifact(
        best_ckpt_artifact,
        target_path=target_path,
        aliases=["production", best_alias],
    )
    print(f"linked to {target_path} with aliases: production, {best_alias}")

# `with` exited -> run.finish() was called automatically.


## 5. Wrap up — what to look at in the W&B UI

One run was created by this notebook (`tiny-transolver-train`,
`job_type = train`). In the W&B UI you should see:

**Lineage graph**
- `wandb-registry-airfoil_surrogates/datasets:production` (produced
  by `registry.ipynb`) → this training run → newly-linked
  `wandb-registry-airfoil_surrogates/checkpoints:production`.

**Training run metrics**
- Line charts for `train/loss`, `val/loss`, `val/pressure_rmse`,
  and `val/slice_entropy`.

**Artifacts logged by the training run**
- `airfoil_surrogate_ckpt:v0..v11` — one checkpoint per epoch,
  aliased `epoch-N`.

**Media panels**
- `val/predictions_epoch_*` Tables with inline pressure-field images
  (click a row to expand the image).
- `val/rmse_by_aoa_epoch_*` bar charts.
- `model/slice_assignment_epoch_*` images (drag through the epochs to
  watch the slicing evolve).

**Registry**
- Registry `airfoil_surrogates` now has two Collections:
  - `datasets` (from `registry.ipynb`).
  - `checkpoints` — best-val model version aliased `production` and
    `epoch-N`.

Run `sweeps.ipynb` next to fine-tune the champion checkpoint.
